# 05 · The geometry of superposition

**Paper:** "The Geometry of Superposition": Uniform Superposition, Feature Dimensionality, Why
These Geometric Structures?, Polytopes and Low-Rank Matrices (pp. 16–21), and Non-Uniform
Superposition (pp. 21–25).
**Experiments:** E03 `uniform_geometry`, E04 `nonuniform_geometry`.

**Learning goals**

1. Compute D* = m/‖W‖²_F and each feature's dimensionality Dᵢ by hand, and name a tegum factor
   by how many features it holds and how many dimensions it spans.
2. Read E03's curves: D* sticks at 1 and at 1/2, and the Dᵢ cluster at fractions that belong to
   uniform polytopes (pp. 17–19).
3. Watch non-uniform features deform a polytope until it snaps to another (p. 22), and see how
   correlated and anticorrelated features arrange themselves (pp. 23–25).

On a laptop CPU the E03 quick profile takes about 6 minutes and E04's about
5 minutes.

In [ ]:
# Runs locally and in Colab. In a fresh environment (e.g. Colab), install the package first.
# Replace <OWNER> with the GitHub owner of your copy of the repository.
try:
    import superposition  # noqa: F401
except ImportError:
    %pip install -q "git+https://github.com/<OWNER>/superposition.git"

In [ ]:
import math

import pandas as pd
import torch
import yaml
from IPython.display import Image, display

from superposition import metrics
from superposition.experiments.runner import run_experiment
from superposition.runs import list_runs

REUSE = True  # reuse your latest quick-profile run if there is one; False always re-runs


def quick_run(name: str):
    """Your latest complete quick-profile run of ``name`` with its figures, or a fresh one.

    Runs made without figures (``superposition run --no-figures``, or the replication tests)
    are skipped: this notebook shows the run's figures.
    """
    if REUSE:
        for run in reversed(list_runs(name)):
            quick = yaml.safe_load(run.config_path.read_text())["profile"] == "quick"
            if quick and any(run.figures_dir.glob("*.png")):
                print(f"reusing run {run.run_id}")
                return run
    return run_experiment(name, "quick", device="auto")


def criteria(run) -> pd.DataFrame:
    doc = run.read_acceptance()
    print("overall:", doc["overall"], f"({doc['n_passed']}/{doc['n_graded']} graded criteria)")
    return pd.DataFrame(doc["criteria"])[["group", "id", "status", "value"]]


def polygon(k: int) -> torch.Tensor:
    """k unit vectors evenly spaced on the circle, shape (2, k)."""
    angles = 2 * math.pi * torch.arange(k) / k
    return torch.stack([angles.cos(), angles.sin()])


def describe(W: torch.Tensor) -> pd.Series:
    """Dᵢ of every feature and the name of each tegum factor, for one model (hidden × feat)."""
    dims = metrics.feature_dimensionality(W[None])[0].numpy().round(3)
    return pd.Series(
        {
            "Dᵢ": dims.tolist(),
            "Σ Dᵢ": round(float(dims.sum()), 3),
            "D*": round(float(metrics.dims_per_feature(W[None])[0]), 3),
            "tegum factors": metrics.geometry_census(W),
        }
    )

## 1. Dimensionality by hand (pp. 18–21)

Dᵢ = ‖Wᵢ‖² / Σⱼ (Ŵᵢ·Wⱼ)² asks what fraction of a dimension feature i gets: the numerator is how
much of it is stored, the denominator how many features share its direction. **Predict** Dᵢ for
a pentagon in 2D, a tetrahedron in 3D, and a triangular bipyramid: a triangle in 2D plus an
antipodal pair on the third axis.

In [ ]:
tetrahedron = torch.tensor([[1, 1, 1], [1, -1, -1], [-1, 1, -1], [-1, -1, 1]]).T / math.sqrt(3)
bipyramid = torch.block_diag(polygon(3), torch.tensor([[1.0, -1.0]]))  # a tegum product (p. 20)
pd.DataFrame(
    {
        "pentagon": describe(polygon(5)),
        "tetrahedron": describe(tetrahedron.float()),
        "triangular bipyramid": describe(bipyramid),
    }
).T

A uniform polytope gives every vertex the same Dᵢ = (dimensions spanned) / (features), and the
Dᵢ of all represented features add up to the dimensions used. The bipyramid is not uniform: it is
the tegum product of a triangle and a digon in orthogonal subspaces, and its features keep their
factors' values, 2/3 and 1/2, rather than sharing a single 3/5 (p. 19). Naming each connected
factor by (features, dimensions) is how E03 tells a pentagon (2/5) from a square antiprism
(3/8), which are only 0.025 apart in Dᵢ (docs/DECISIONS.md D-308).

## 2. Uniform superposition: E03 (pp. 17–20)

n = 400 equally important, equally sparse features in m = 30 dimensions, with 1/(1 − S) from 1 to
10. **Predict:** how many features does the model represent when D* = 1/2? What happens to D* as
features get sparser?

In [ ]:
e03 = quick_run("uniform_geometry")
criteria(e03)

In [ ]:
for name in ("dstar", "dimensionality", "geometry_graphs", "factor_census"):
    display(Image(filename=str(e03.figures_dir / f"{name}.png")))

D* sits at 1 while features are dense, on the 1/2 plateau from about 1/(1 − S) = 2.3 to 3.8,
and falls to about 0.26 at 8, as on p. 17. In the Dᵢ scatter the plateau's features sit at 1/2,
and there are clusters near 2/3, 2/5 and 3/8; beyond about 5 the Dᵢ form a band that falls with
D*, as in the paper's p. 19 figure. There is no 3/4 (tetrahedron) cluster, the clearest
difference from p. 19 (docs/REPRODUCTION.md, E03). And the Dᵢ add up to m = 30 at every
sparsity (p. 18):

In [ ]:
sums = next(c for c in e03.read_acceptance()["criteria"] if c["id"] == "sum_dimensionality")
pd.DataFrame(sums["value"]).T.rename_axis("1/(1 − S)").round(3)

**Which polytopes are really there?** A Dᵢ near 3/8 does not by itself show a square antiprism,
and the geometry graph only separates tegum factors that are exactly orthogonal. E03 joins two
features when |Wᵢ·Wⱼ| > 0.05 (docs/DECISIONS.md D-110). Trained factors are only *nearly*
orthogonal, so at high sparsity a few small overlaps can link everything into one component.
The table counts the represented features that sit in one of the paper's named polytopes, for
three edge thresholds. **Predict:** at which sparsities do stricter thresholds reveal more
named polytopes?

In [ ]:
from superposition.experiments.common import GridResult

grid = GridResult.from_checkpoint(e03.load_checkpoint("relu_output"))
named = set(metrics.POLYTOPES.values())
rows = []
for cond in range(grid.n_conditions):
    W, _, _ = grid.best_weights(cond)
    row = {"1/(1 − S)": round(1 / grid.condition_probability(cond), 2)}
    for edge in (0.05, 0.1, 0.2):
        census = metrics.geometry_census(W, edge_threshold=edge)
        row["represented"] = sum(census.values())
        row[f"in named polytopes (edge {edge})"] = sum(v for k, v in census.items() if k in named)
        if edge == 0.2:
            row["factors at edge 0.2"] = dict(sorted(census.items(), key=lambda kv: -kv[1])[:4])
    rows.append(row)
pd.set_option("display.max_colwidth", 140)
pd.DataFrame(rows)

With the default threshold only the plateau's digons (and one pentagon) are separate factors.
Ignoring overlaps below 0.2, the model at 1/(1 − S) ≈ 4.3 is made entirely of digons and
pentagons, and around 4.8–5.5 pentagons, digons and square antiprisms appear: the p. 19
polytopes, as factors that are nearly rather than exactly orthogonal. Sparser still, one
component spans all 30 dimensions at every threshold, so the falling band of the scatter plot is
not made of the paper's polytopes.

## 3. Non-uniform superposition: E04 (pp. 21–25)

**(a)** Five features in two dimensions at density 0.05 make a pentagon. Feature 0's density is
then scaled from 0.1× to 10×. **Predict:** as feature 0 gets denser, do its neighbours move toward
it or away? What happens when it gets much sparser?

In [ ]:
e04 = quick_run("nonuniform_geometry")
criteria(e04)

In [ ]:
for name in ("perturbed_solutions", "perturbed_losses"):
    display(Image(filename=str(e04.figures_dir / f"{name}.png")))

Denser, feature 0's neighbours move away (the angle next to it grows past 72°); sparser, they
close in; and at about a quarter of the others' density the best solution snaps to two digons
with feature 0 dropped. The loss plot shows why this is a first-order change: each geometry has
its own loss curve, and the best solution switches where two curves cross. Our snap comes at a
lower relative density than the paper's "≈ 2.5× less dense" (docs/REPRODUCTION.md, E04).

**(b, c)** Correlated sets switch on together; in an anticorrelated set at most one member is
active (p. 23). **Predict:** in two dimensions, should two correlated features be orthogonal,
antipodal or side by side? And two anticorrelated ones?

In [ ]:
display(Image(filename=str(e04.figures_dir / "correlated_sets.png")))
display(Image(filename=str(e04.figures_dir / "local_bases_gram.png")))

Each correlated pair is orthogonal, each anticorrelated pair antipodal, and three correlated
pairs sit side by side, as on p. 24. The antipodal anticorrelated solution has essentially zero
loss, since the two members are never active together, but only a minority of restarts find it
(the INFO row `within_set_cos_all_restarts` lists every restart). In the larger model, WᵀW is
nearly diagonal within each set of 10 and not across the sets: each correlated set forms a local
almost-orthogonal basis (p. 25).

**(d)** Three correlated pairs in two dimensions, swept from very sparse to dense. **Predict:**
which configuration do you expect at the sparse end, and which at the dense end (p. 25)?

In [ ]:
display(Image(filename=str(e04.figures_dir / "collapse.png")))

Sparse: a hexagon with each pair side by side. Denser: the pairs collapse one after another onto
their principal component, drawn at radius about 1/√2 as on p. 25. Densest: all three have
collapsed. The regimes sit at somewhat higher densities than p. 25 gives
(docs/REPRODUCTION.md, E04).

## Exercises

**1.** Take E03's best model at the grid point nearest 1/(1 − S) = 3. How many features does it
represent, how many tegum factors of each kind does it have, and how does that relate to D*?

<details><summary>Solution</summary>

```python
import numpy as np

grid = GridResult.from_checkpoint(e03.load_checkpoint("relu_output"))
x = np.array([1 / grid.condition_probability(c) for c in range(grid.n_conditions)])
point = int(np.argmin(np.abs(x - 3.0)))
W, _, _ = grid.best_weights(point)
print(f"1/(1 − S) = {x[point]:.2f}, represented: {int(metrics.represented_mask(W[None])[0].sum())}")
print("D* =", round(float(metrics.dims_per_feature(W[None])[0]), 3))
print(metrics.geometry_census(W))
```

On the 1/2 plateau nearly every represented feature sits in a digon: 30 dimensions hold about
60 features, and D* = m / ‖W‖²_F ≈ 30/60.
</details>

**2.** Show that a square antiprism (two unit squares at heights ±1/√3, turned 45° apart)
gives every vertex Dᵢ = 3/8. What happens when both squares move to heights ±0.4, and when they
sit at unequal heights, +0.7 and −0.3?

<details><summary>Solution</summary>

```python
def antiprism(top, bottom):
    angles = torch.arange(8) * math.pi / 4
    z = torch.tensor([top if k % 2 == 0 else -bottom for k in range(8)])
    radius = torch.sqrt(1 - z**2)  # unit vectors
    return torch.stack([radius * angles.cos(), radius * angles.sin(), z]).float()


for top, bottom in ((1 / math.sqrt(3), 1 / math.sqrt(3)), (0.4, 0.4), (0.7, 0.3)):
    W = antiprism(top, bottom)
    dims = metrics.feature_dimensionality(W[None])[0]
    name = metrics.component_geometry(W, list(range(8)))[2]
    print(f"±({top:.3f}, {bottom:.3f}): Dᵢ {dims.numpy().round(3)}, Σ {float(dims.sum()):.2f}, {name}")
```

At ±1/√3 the eight unit vectors form a tight frame (Σ WᵢWᵢᵀ = 8/3 · I), which makes every
Dᵢ = m/n = 3/8 and Σ Dᵢ = m = 3. At ±0.4 every vertex still looks the same, so the Dᵢ are
still equal, but the frame is no longer tight: each Dᵢ drops to about 0.33 and they no longer
add up to m (p. 18's "packed efficiently"). At unequal heights the two squares get different
Dᵢ. `component_geometry` names all three "square antiprism": its uniformity test allows Dᵢ
within 0.05 of r/k (docs/DECISIONS.md D-308), so it names shapes, not only tight ones.
</details>

**3.** In E04 (a), at which relative density does the best solution switch from the pentagon to
two digons? Compare with the paper's "≈ 2.5× less dense than the others" (p. 22).

<details><summary>Solution</summary>

```python
value = next(c for c in e04.read_acceptance()["criteria"] if c["id"] == "collapse_to_digons")
print(value["value"]["best_geometry_by_ratio"])
print("largest ratio with the collapse:", value["value"]["largest_ratio_with_collapse"])
```
</details>

**4.** Why do anticorrelated features like to be antipodal? Compute the 1-sparse interference of
an antipodal pair (it is zero), then think about what happens when both members are active at
once, and why that never happens in an anticorrelated set.

<details><summary>Solution</summary>

With W = [1, −1] and b = 0, output 0 is ReLU(x₀ − x₁). With only one feature active the other
output sees a negative pre-activation, clipped to exactly zero error (notebook 03). With both
active, x₀ − x₁ underestimates x₀: a correlated pair pays that price every time it fires, an
anticorrelated pair never does. So anticorrelated features get antipodal pairs for free, while
correlated ones prefer to be orthogonal, and then side by side.
</details>